[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/symmetry/symmetry.ipynb)

# What Symmetry Allows

This example asks which material responses a symmetric object can have. A crystal's properties must look the same after any of its symmetries, so averaging a measured response over the crystal's symmetry group keeps exactly the part the crystal allows. Heat conduction in a cubic crystal comes out the same in every direction, yet its elastic stiffness keeps a cubic shape; and a crystal can rotate polarized light only if it has no mirror or centre of inversion, unless its mirrors exchange directions with opposite senses.

Each response is a map: a conductivity from driving fields, read by crossing them, to heat flows along directions, each principal axis reading the field by their pairing, `field & axis`; an elasticity a form on four directions; a rotary power a form on a direction twice with an intrinsic, handed value. A symmetry moves a map as it moves anything, `rotor >> conductivity(rotor << Field)`, and averaging the moved maps over a group projects the response onto the invariant ones. The rotary power has a sense, so an improper rotation, an odd versor, reverses it where it leaves the conductivity alone.

In the notation of crystal physics the averaging reads as Neumann's principle, the conductivity as a symmetric second-rank tensor $K_{ij}$, the stiffness as the fourth-rank tensor $C_{ijkl}$, and the rotary power as the gyration tensor $g_{ij}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Extensor, concatenate
from numga import NumpyContext
from examples.animation import save_figure
from examples.mechanics.symmetry import render
from examples.mechanics.symmetry.core import cube_rotations, directions, fields, ga, lattice_samples, turns

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
# A driving field, the temperature's fall per unit length, is read by crossing it: a vector. Heat flows,
# bonds and the paths of light are carried by geometry: directions, antivectors.
Field = ga.gatype.vector()
Direction = ga.gatype.antivector()
Conductivity = ga.gatype((Direction, Field))       # Direction <- Field
Rotor = ga.gatype.rotor()
ImproperRotation = ga.gatype.odd()

## 1. Heat conduction

A measured conductivity, positive, with three different gains along a tilted frame of directions, is averaged over three groups: half turns about x and z, quarter turns about z, and the rotations of a cube. Each panel is the image of the unit ball of driving fields, a heat-flow ellipsoid, with one driving field, dark, drawn at the largest gain, and the heat flow it drives, orange. Half turns align the ellipsoid with the axes; quarter turns about z make it round about z; the cube makes it a sphere, and heat then flows along the driving field. The measured conductivity has six independent components, and the three groups leave three, two and one.

In [ ]:
# The measured responses' tilted principal frame, the conductivity's gains along it, and the driving field drawn.
axes = (mv.xy * 0.3 + mv.yz * 0.16).exp() >> Extensor.stack((mv.x.dual(), mv.y.dual(), mv.z.dual()))   # [3] Direction
gains = np.array([4.5, 1.5, 0.6])
driving = (mv.x + mv.y + mv.z).normalized()                                    # [] Field

In [ ]:
def conductivities(axes: Direction, gains: np.ndarray, groups: list[Rotor]) -> Conductivity:
    """Which heat-conduction responses are compatible with a crystal's symmetry?

    Conductivity maps the driving field, the temperature's fall, to heat flow. In an anisotropic
    material the flow need not run along the field. Start with a positive conductivity,
    rotate the whole response through each symmetry, and average. This Reynolds
    projection gives the closest invariant response in the Frobenius norm, preserving
    positive definiteness. Symmetry constrains the response, not its numerical gains.

    Returns the measured conductivity followed by its average over each group.
    """
    # Each principal axis reads the driving field by their pairing and contributes heat flow along
    # itself. Positive gains make this a passive conductivity.
    conductivity = (axes * (Field & axes) * gains).sum(axis=0)                     # Direction <- Field
    # Pull the field into each rotated frame, apply the conductivity, and turn the flow back. The mean is
    # unchanged by any rotation in the group: those rotations merely permute the terms. This projects a
    # measured response onto the allowed ones.
    return Extensor.stack([conductivity] + [(group >> conductivity(group << Field)).mean(axis=0) for group in groups])


unit_fields, sphere = fields(), directions()                                   # [latitudes, longitudes] Field, Direction
groups = [(turns(mv.yz, 2)[:, None] * turns(mv.xy, 2)[None, :]).reshape(-1), turns(mv.xy, 4), cube_rotations()]
responses = conductivities(axes, gains, groups)                                # [candidate and groups] Direction <- Field
surfaces = responses[:, None, None](unit_fields[None, :, :])                    # [candidate and groups, latitudes, longitudes] Direction
fluxes = responses(driving)                                                    # [candidate and groups] Direction
save_figure(render.draw_conduction(responses, driving, fluxes), "symmetry_conductivity");

In [ ]:
# The unit driving field read on the flow, over the flow's length: the cosine of the angle between them.
deflection = np.degrees(np.arccos(((driving & fluxes) / fluxes.norm()).to_array()))
print("angle between the driving field and the heat flow, candidate and each group:", deflection)
print("the cube's conductivity:\n", responses[-1].coefficient_table())

## 3. One group, two responses

A simple-cubic lattice has axial and face-diagonal bonds, directions, and the cube's 24 rotations turn one seed bond of each kind into its whole family. Averaged over them, a bond's conduction, the driving field paired with the bond and returned along it, is the same in every direction, 2/3. Its elasticity, the bond measured against a direction four times by the metric, `bond.dot(direction)`, is not: 1/2 along the cube's axes and 1/3 along its body diagonals. Left, the lattice with its axial bonds, orange, and diagonal bonds, blue; middle, the conduction, the image of the unit ball of fields, a sphere; right, the stiffness along every direction as a radius.

In [ ]:
def lattice_responses(seeds: Direction, cube: Rotor) -> tuple[Conductivity, Extensor]:
    """Average microscopic bond responses over a crystal's cubic point group.

    A simple-cubic lattice has axial and face-diagonal bond families. Starting with
    `mv.x.dual()` and `(mv.x + mv.y).dual().normalized()`, the 24 cube rotations generate every
    direction in each family. Every member has the same weight, so another cube
    rotation just permutes the terms in the average. This is the microscopic meaning
    of the group projection.

    Give the two families equal total response weights. For the elastic model, take
    unstressed central springs with lattice spacing 1, axial stiffness 1/3, and
    face-diagonal stiffness 1/12. Bond counts and squared lengths give weight 1 to
    each family. Diagonal springs provide shear stiffness. This particular model has
    equal elastic constants c12 and c44; a general cubic crystal need not have that
    extra relation.

    A unit bond n reads a driving field f through their pairing, `f & n`. For an imposed
    strain of size s along a unit direction d, its fractional extension is `s * n.dot(d) ** 2`,
    measured by the metric between the two directions. Squaring to get spring energy gives
    four factors of `n.dot(d)`, hence a scalar form with four direction inputs. Its value
    `elasticity(d, d, d, d)` is the stiffness, with energy density
    `s ** 2 * elasticity(d, d, d, d) / 2`. It holds transverse strain fixed; it is not
    the relaxed Young's modulus.

    Cubic elasticity background: https://www.ctcms.nist.gov/oof/oof1/Manual/node152.html
    Returns the conductivity and the elasticity.
    """
    # Batch axes are [24 rotations, 2 bond families]. These are free bond directions;
    # the actual lattice sites are projective points.
    bonds = cube[:, None] >> seeds[None, :]                                      # [rotations, families] Direction

    # Average over the group, then add the two families. No direction is privileged within either
    # orbit. Conductivity reads the field along each bond and returns flow along it.
    conductivity = (bonds * (Field & bonds)).mean(axis=0).sum(axis=0)            # Direction <- Field
    projection = bonds.dot(Direction)                                            # [rotations, families] Scalar <- Direction

    # Four independent inputs encode the spring stiffness. The same group average
    # leaves cubic structure here, although it made the conduction isotropic.
    elasticity = (projection * projection * projection * projection).mean(axis=0).sum(axis=0)
    return conductivity, elasticity


seeds = mv.vector([[1.0, 0.0, 0.0], [1.0, 1.0, 0.0]]).dual().normalized()      # [families] Direction
sites, axial, diagonal = lattice_samples()
conductivity, elasticity = lattice_responses(seeds, cube_rotations())
stiffness_surface = sphere * elasticity(sphere, sphere, sphere, sphere)         # [latitudes, longitudes] Direction
save_figure(render.draw_crystal(sites, axial, diagonal, conductivity, stiffness_surface), "symmetry_crystal_lattice");

In [ ]:
axis_direction, body_diagonal = mv.x.dual(), (mv.x + mv.y + mv.z).dual().normalized()   # [] Direction
print("stiffness along an axis and along a body diagonal:", elasticity(*[axis_direction] * 4).to_array(), elasticity(*[body_diagonal] * 4).to_array())

## 4. Optical activity

A rotary power turns light's polarization by an amount with a sense, so it is an intrinsic, handed scalar, measured against the lab's handedness. A point group holds rotations and, if the crystal has mirrors or a centre of inversion, improper rotations, which reverse that sense. Averaged over the rotations of a square, two rotary powers remain; with the inversion, none. Quarter turns through the inversion leave opposite senses along x and y and none along z: such a crystal is its own mirror image, yet turns light. The cube's rotations leave one rotary power, the same in every direction, as in a sugar solution; with the inversion, none. Each sphere is the sphere of light directions, coloured by the rotary power along each, red one sense and blue the other: the candidate, then the five groups.

In [ ]:
powers = np.array([3.0, -1.0, 0.5])

In [ ]:
def rotary_powers(axes: Direction, powers: np.ndarray, groups: list[tuple[Rotor, ImproperRotation]]) -> Extensor:
    """Which crystals can rotate polarized light?

    Light crossing an optically active crystal along a unit direction `d` has its plane of polarization
    turned, in proportion to the distance, by the rotary power `gyration(d, d)`. The turn has a sense,
    so the rotary power is an intrinsic scalar: the mirror image of a crystal turns light the other way.
    Its principal values are measured against the lab's handedness, the volume `x ^ y ^ z` read as a
    number.

    A crystal's point group holds its rotations and, where the crystal has mirrors or a centre of
    inversion, its improper rotations, the odd versors. An improper rotation reverses the sense of an
    intrinsic output, so a group allows less optical activity than its rotations alone, and none where
    it contains the inversion.

    Returns the candidate followed by its average over each group.
    """
    handedness = mv.xyz.dual()                                                   # [] Scalar, intrinsic
    gyration = (axes.dot(Direction) * axes.dot(Direction) * powers).sum(axis=0) * handedness   # Scalar <- (Direction, Direction)

    def average(rotations: Rotor, improper: ImproperRotation) -> Extensor:
        # Turn both directions into each element's frame, and carry the rotary power back out: an improper
        # rotation reverses its sense.
        return concatenate([element >> gyration(element << Direction, element << Direction) for element in (rotations, improper)]).mean(axis=0)

    return Extensor.stack([gyration] + [average(rotations, improper) for rotations, improper in groups])


inversion = mv.xyz
quarter, cube = turns(mv.xy, 4), cube_rotations()
point_groups = [(quarter, inversion * quarter[:0]), (quarter, inversion * quarter), (quarter[::2], inversion * quarter[1::2]),
                (cube, inversion * cube[:0]), (cube, inversion * cube)]
rotary = rotary_powers(axes, powers, point_groups)                             # [candidate and groups] Scalar <- (Direction, Direction)
rotary_surfaces = rotary[:, None, None](sphere[None, :, :], sphere[None, :, :])
save_figure(render.draw_optical_activity(rotary), "symmetry_optical_activity");

In [ ]:
print("largest rotary power left, per group:", rotary_surfaces[1:].norm().to_array().max(axis=(1, 2)))

In [ ]:
# checks
# The cube's conductivity is the mean gain in every direction; lattice conduction is isotropic, stiffness
# cubic; the inversion removes every rotary power, and the cube's rotations leave the mean one.
assert (surfaces[-1] - sphere * gains.mean()).norm().to_array().max() < 1e-7
np.testing.assert_allclose((unit_fields & conductivity(unit_fields)).to_array(), 2 / 3, rtol=1e-7)
np.testing.assert_allclose(elasticity(*[axis_direction] * 4).to_array(), 1 / 2, rtol=1e-7)
np.testing.assert_allclose(elasticity(*[body_diagonal] * 4).to_array(), 1 / 3, rtol=1e-7)
for centrosymmetric in (rotary_surfaces[2], rotary_surfaces[5]):
    assert centrosymmetric.norm().to_array().max() < 1e-12
assert (rotary_surfaces[4] - mv.xyz.dual() * powers.mean()).norm().to_array().max() < 1e-12